# 01 — Data Engineering & Preprocessing (UBCF)

Phase 1 (user-based adaptation):
- Chunked ingestion of MovieLens 32M ratings
- ID remapping and user-focused sparse matrix
- Truncated SVD (32 latent user features)
- Content features from genres/tags for hybrid fallback

Outputs: `user_sparse.npz`, `user_latent.npy`

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.decomposition import TruncatedSVD

DATA_DIR = Path("data/ml-32m")
OUT_DIR = Path("data/processed_32m_ubcf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RATINGS_PATH = DATA_DIR / "ratings.csv"
MOVIES_PATH = DATA_DIR / "movies.csv"
TAGS_PATH = DATA_DIR / "tags.csv"

CHUNK_SIZE = 2_000_000
N_COMPONENTS = 32
SEED = 42
TOP_TAGS = 120

print(RATINGS_PATH, MOVIES_PATH, TAGS_PATH)

data\ml-32m\ratings.csv data\ml-32m\movies.csv data\ml-32m\tags.csv


In [2]:
# Chunked pass for ID discovery
all_users, all_movies = set(), set()
n_rows = 0

for chunk in pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"], chunksize=CHUNK_SIZE):
    n_rows += len(chunk)
    all_users.update(chunk["userId"].unique().tolist())
    all_movies.update(chunk["movieId"].unique().tolist())

unique_users = np.array(sorted(all_users), dtype=np.int64)
unique_movies = np.array(sorted(all_movies), dtype=np.int64)
user_to_idx = {u: i for i, u in enumerate(unique_users)}
movie_to_idx = {m: i for i, m in enumerate(unique_movies)}

print(f"rows={n_rows:,} users={len(unique_users):,} movies={len(unique_movies):,}")

rows=32,000,204 users=200,948 movies=84,432


In [3]:
# Build user-focused CSR matrix: rows=users, cols=movies
row_parts, col_parts, val_parts = [], [], []
for chunk in pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"], chunksize=CHUNK_SIZE):
    row_parts.append(chunk["userId"].map(user_to_idx).to_numpy(dtype=np.int32, copy=False))
    col_parts.append(chunk["movieId"].map(movie_to_idx).to_numpy(dtype=np.int32, copy=False))
    val_parts.append(chunk["rating"].to_numpy(dtype=np.float32, copy=False))

rows = np.concatenate(row_parts)
cols = np.concatenate(col_parts)
vals = np.concatenate(val_parts)
user_sparse = csr_matrix((vals, (rows, cols)), shape=(len(unique_users), len(unique_movies)), dtype=np.float32)

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)
user_latent = svd.fit_transform(user_sparse).astype(np.float32)

print(user_sparse.shape, user_sparse.nnz)
print(user_latent.shape)

(200948, 84432) 32000204
(200948, 32)


In [4]:
# Content features from movies + tags for hybrid cold-start fallback
movies = pd.read_csv(MOVIES_PATH, usecols=["movieId", "genres"])
movies = movies[movies["movieId"].isin(unique_movies)].copy()
movies["genres"] = movies["genres"].fillna("(no genres listed)")
genre_df = movies["genres"].str.get_dummies(sep="|")
genre_df.index = movies["movieId"].map(movie_to_idx)
genre_df = genre_df.sort_index()

tags = pd.read_csv(TAGS_PATH, usecols=["movieId", "tag"])
tags = tags[tags["movieId"].isin(unique_movies)].copy()
tags["tag"] = tags["tag"].astype(str).str.lower().str.strip()
tags = tags[tags["tag"] != ""]

if len(tags) > 0:
    top_tags = tags["tag"].value_counts().head(TOP_TAGS).index
    tags = tags[tags["tag"].isin(top_tags)]
    tags["v"] = 1
    tag_df = tags.pivot_table(index="movieId", columns="tag", values="v", aggfunc="max", fill_value=0)
    tag_df.index = tag_df.index.map(movie_to_idx)
    tag_df = tag_df.sort_index()
else:
    tag_df = pd.DataFrame()

full_idx = np.arange(len(unique_movies), dtype=np.int32)
genre_df = genre_df.reindex(full_idx, fill_value=0)
tag_df = tag_df.reindex(full_idx, fill_value=0) if not tag_df.empty else pd.DataFrame(index=full_idx)
content_features = pd.concat([genre_df, tag_df], axis=1).astype(np.float32).to_numpy(copy=False)

save_npz(OUT_DIR / "user_sparse.npz", user_sparse)
np.save(OUT_DIR / "user_latent.npy", user_latent)
np.save(OUT_DIR / "content_features.npy", content_features)
with open(OUT_DIR / "id_maps.pkl", "wb") as f:
    pickle.dump({"unique_users": unique_users, "unique_movies": unique_movies, "user_to_idx": user_to_idx, "movie_to_idx": movie_to_idx}, f)

print("Saved: user_sparse.npz, user_latent.npy, content_features.npy, id_maps.pkl")

Saved: user_sparse.npz, user_latent.npy, content_features.npy, id_maps.pkl
